# Анализ неструктурированных данных. Задание. Распознавание цифр. Вариант 5

Выполнил студент 325 группы Жигалов Никита Сергеевич

## Постановка (README)

Я решил строить сеть для распознования 5 различных букв латинского алфавита: N, K, P, R, Z.

Для этого был создан датасет `letters_dataset`, в котором содержится по 10 вручную созданных примеров на каждую букву. Файлы имеют разрешение 32 на 32, и формат `.png`, создать свои примеры (*бесплатно*) можно на сайте [piskel](https://www.piskelapp.com/p/create/sprite), главное выставить соответствующие настройки.

Датасет `augmented_dataset` - содержит дополнительную аугментацию букв, содержащихся в `letters_dataset`, она заключается в сдвиге данных по картинке и рисовании букв различными цветами.

В папку `models` сохранялись рабочие модели для последующего их использования (чтобы не тратить время на обучение при повторных запусках ноутбука)

В папке `test_inputs` лежат 10 тестовых примеров различных букв (созданных отдельно, они не встречаются в тренировочном датасете), сюда же необходимо добавлять новые примеры при желании дополнительно протестировать модель.

Для корректной работы программы необходимо, наличие всех папок, не перемещайте этот ноутбук в другие директории!

При работе использовались библиотеки:

* pillow (PIL), а конкретно Image, для конвертации `.png` файлов в формат numpy-массивов, а также для вывода их на экран
* numpy - для работы с массивами пикселей, представляющих картинки
* pandas - для ковертации массивов данных в удобный формат DataFrame, и последующего его использования для разделения данных на тренировочную и обучающие выборки
* tensorflow, а конкретно keras, для построения нейронной сети, осуществляющей обработку изображений
* os - для чтения файлов
* matplotlib.pyplot - для вывода рисунков символов на экран
* train_test_split из sklearn для удобного (случайного) разбиения данных на обучающую и тестовую выборки

## Подготовка данных для работы

Подключаем необходимые библиотеки

In [ ]:
from PIL import Image
import numpy as np
import pandas as pd
from tensorflow import keras
from tensorflow.keras import layers
import os
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

Получаем расположение файла ноутбука

In [ ]:
path = os.getcwd()

Создаем метки классов

In [ ]:
labels = ['K', 'N', 'P', 'R', 'Z'] # метки для картинок
y_not_augm = np.repeat(np.arange(5), 10) # соответствующий числовой эквивалент метки (от 0 до 4)
y_augm = np.repeat(np.arange(5), 15) # метки аугментированных данных
y_not_augm

In [ ]:
col_names = ['pxl' + str(i) for i in range(1024)] # задаем названия колонок в датафрейме
# всего пикселей 32 * 32 = 1024

Заполняем DataFrame для дальнейшего разбиения на train и test

In [ ]:
# сначала неаугментированные данные
df = pd.DataFrame(columns=col_names)
num_classes = 5 # число классов
samples_per_class = 10 # число объектов в классе
plt.figure(figsize=(14, 10))
for j, l in enumerate(labels):
    for i in range(1, 10 + 1):
        image = Image.open(path + '/letters_dataset/' + l + '_' + str(i) + '.png') # открываем изображения        
        new_image = image.convert('L')  # превращаем картинку в черно-белую (для уменьшения размерности)
        plt.subplot(num_classes, samples_per_class, j * 10 + i) # вывод всех рисунков на одном полотне
        plt.imshow(np.asarray(new_image), cmap='gray', vmin=0, vmax=255)
        plt.axis('off')
        numpydata = np.asarray(new_image).flatten() # преобразуем пиксели в числовой формат
        df.loc[l + '_' + str(i)] = numpydata  # добавляем новую строку в датафрейм
                

In [ ]:
lines_num = 8 # число классов
row_size = 10 # число объектов в классе
plt.figure(figsize=(14, 10))
for j, l in enumerate(labels):
    for i in range(1, 5 + 1):
        for q in range(1, 3 + 1):
            image = Image.open(path + '/augmented_dataset/' + l + '_' + str(i) + '_' + str(q) + '.png') # открываем изображения        
            new_image = image.convert('L')  # превращаем картинку в черно-белую (для уменьшения размерности)
            plt.subplot(lines_num, row_size, j * 15 + (i-1) * 3 + q) # вывод всех рисунков на одном полотне
            plt.imshow(image)
            plt.axis('off')
            numpydata = np.asarray(new_image).flatten() # преобразуем пиксели в числовой формат
            df.loc[l + '_' + str(i) + '_' + str(q)] = numpydata  # добавляем новую строку в датафрейм

Также можем наблюдать все рисунки букв, входящих в наш тренировочный датасет

In [ ]:
y = np.hstack((y_not_augm, y_augm))
y

In [ ]:
df.insert(0, "letter", y, allow_duplicates=True)

Разбиваем данные на train и test

In [ ]:
data = np.array(df)
X_train, X_test, y_train, y_test = train_test_split(data[:, 1:], data[:, 0], random_state=42, test_size=0.2, stratify=y)

In [ ]:
data.shape

In [ ]:
X_train.shape, y_train.shape, X_test.shape, y_test.shape

Приводим значения пикселей к [0, 1], а метки y_train и y_test к категориальным переменным 

In [ ]:
X_train = X_train.astype('float32') / 255
X_test = X_test.astype('float32') / 255

y_train_categorical = keras.utils.to_categorical(y_train, 5)
y_test_categorical = keras.utils.to_categorical(y_test, 5)

На этом моменте предобработка данных завершена, в таком виде они будут подаваться на вход для обучения и тестирования моделей, как будет показано во второй части задания при построении модели

## Проверка работы модели на пользовательских тестах

В ячейке ниже загружается лучший пример модели, если хотите использовать модель полученную на предыдущих шагах, то пропустите выполнение этой ячейки

In [ ]:
my_nn = keras.models.load_model(path + '/models/m5.keras')

In [ ]:
loss, accuracy = my_nn.evaluate(X_test.reshape(-1, 32, 32, 1), y_test_categorical)
print(f'Loss: {loss}, accuracy: {accuracy}')

Пример ниже - шаблонный, если хотите добавить свои данные то необходимо: 
1. Формат файла `.png`, размер в пикселях 32 на 32 
2. Это нужно делать в папку `test_inputs`, обозначая примеры последовательно, например, если добавлять новые тестовые символы к текущим десяти, то следующий файл должен иметь название `letter_11.png`, потом `letter_12.png`, и т.д.
3. Если нужен вывод на экран тестовых данных, то следует поменять переменные `num_of_samples` на новое число входных данных, а `split_to_rows` на число рядов в которых они будут выводится, `num_of_samples` должно быть кратно `split_to_rows`!!! Иначе возникнет ошибка.
4. В противном случае для быстрой проверки, можно поменять значение переменной `make_plot` на `False`, тогда изображения выводится не будут, но модель все равно отобразит свои предсказания
5. Создание новых примеров - [piskel](https://www.piskelapp.com/p/create/sprite)


Два подготовленных варианта входных данных:
* просто 10 примеров по 2 каждой буквы
* 15 аугментированных вариаций буквы K
Для использования первого варианта - загрузить следующую ячейку, для второго следующую после него

Первый вариант:

In [ ]:
num_of_samples = 10 # число входных данных
split_to_rows = 2 # число строк при выводе картинок
in_path = '/test_inputs'

Второй вариант:

In [ ]:
num_of_samples = 15 # число входных данных
split_to_rows = 3 # число строк при выводе картинок
in_path = '/augmented_input_K'

In [ ]:
index = 1
# ______________
make_plot = True # изменить на False для быстрого добавления новых данных
# ______________
while os.path.isfile(path + in_path + f"/letter_{index}.png"):
    img = Image.open(path + in_path + f"/letter_{index}.png")
    # здесь конвертируем в ч/б формат и превращаем в формат массива, который можно передать на вход как тренировочные данные
    new_image = np.asarray(img.convert('L'))
    
    if make_plot: # вывод рисунка буквы на одном полотне
        plt.subplot(split_to_rows, num_of_samples // split_to_rows, index)
        plt.imshow(img)
        plt.axis('off')
        
    prediction = my_nn.predict(new_image.reshape(1, 32, 32, 1), verbose=False) # получаем предсказание
    result = labels[np.argmax(prediction)] # достаем соответствующую метку класса
    print(f"{index} The letter is probably {result}")
    index += 1 

if make_plot:
    plt.suptitle('Входные фигуры')
    plt.show()

Как можно видеть для подготовленного примера, модель ошиблась в двух местах, классифицировав R в примере 8, как K и наоборот, в примере 9 классифицировав K как R